# BLM5135 — Track 1 training runbook

Run cell-by-cell or use **Runtime → Run all**. Outputs go directly to your Drive at `/content/drive/MyDrive/dl_project_outputs/`, so if the session dies you can come back and resume any run.

**Before starting:** confirm the runtime is A100 via *Runtime → Change runtime type → A100 GPU*. If only T4/V100/L4 is available, training will work but be slower than the budget assumes.

## 1. Mount Drive and confirm dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
DATASET_ROOT = '/content/drive/MyDrive/dataset'
JSON_PATH = f'{DATASET_ROOT}/dataset.json'
assert os.path.isfile(JSON_PATH), f'dataset.json not found at {JSON_PATH}'
with open(JSON_PATH) as fh:
    nimg = len(json.load(fh)['images'])
print(f'Drive mounted. dataset.json has {nimg} samples.')

## 2. Confirm GPU is A100 (or at least a CUDA device)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Clone the repo (or update if already cloned) and set up output dirs

In [ ]:
import os, subprocess
REPO_URL = 'https://github.com/amirkiarafiei/deep-learning-project.git'
REPO_DIR = '/content/deep-learning-project'
DRIVE_OUT = '/content/drive/MyDrive/dl_project_outputs'
HF_CACHE = f'{DRIVE_OUT}/hf_cache'

if os.path.isdir(f'{REPO_DIR}/.git'):
    subprocess.run(['git', 'fetch', 'origin'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'reset', '--hard', 'origin/main'], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--depth=1', REPO_URL, REPO_DIR], check=True)

for d in [
    DRIVE_OUT, HF_CACHE,
    f'{DRIVE_OUT}/results/track1/object',
    f'{DRIVE_OUT}/results/track1/event',
    f'{DRIVE_OUT}/results/track1/attribute',
    f'{DRIVE_OUT}/results/track1/multitask',
    f'{DRIVE_OUT}/results/track1/object_smoke',
]:
    os.makedirs(d, exist_ok=True)

subprocess.run(['git', 'log', '-1', '--oneline'], cwd=REPO_DIR)
print(f'Repo at {REPO_DIR}, outputs at {DRIVE_OUT}, HF cache at {HF_CACHE}.')

## 4. Install Python deps

Colab usually has a CUDA torch preinstalled — we only need to add `timm` and a few small libs.

In [ ]:
!pip install -q 'timm==1.0.27' 'scikit-learn>=1.4,<2.0' 'PyYAML>=6.0,<7.0' 'tqdm>=4.66'
import torch, timm, sklearn
print('torch  ', torch.__version__, '| cuda:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')
print('timm   ', timm.__version__)
print('sklearn', sklearn.__version__)

## 5. Smoke test — 200 train / 100 val, 2 epochs, batch 32

Should finish in well under 5 minutes on A100. Confirms the full pipeline (data loader, model forward, autocast bf16, loss, optimizer, checkpoint write to Drive) works end-to-end before committing to the 4 real runs.

In [ ]:
import os, subprocess
env = os.environ.copy()
env['HF_HOME'] = HF_CACHE
env['PYTHONUNBUFFERED'] = '1'  # stream subprocess stdout to the cell live
cmd = [
    'python', '-u', 'train.py',
    '--config', 'configs/track1_object.yaml',
    '--output-dir', f'{DRIVE_OUT}/results/track1/object_smoke',
    '--dataset-root', DATASET_ROOT,
    '--json-path', JSON_PATH,
    '--run-name', 'object_smoke',
    '--subset-train', '200',
    '--subset-val', '100',
    '--epochs', '2',
    '--batch-size', '32',
    '--num-workers', '2',
]
subprocess.run(cmd, cwd=REPO_DIR, env=env, check=True)
print('\n✅ Smoke test complete.')

## 6. Full Track-1 training — four configs in sequence

Each: ConvNeXt-Tiny + 30 epochs + early stop on val avg macro-F1 + patience 10. Outputs to Drive every epoch so any session loss bounds the cost to one epoch (resume with `--resume <output_dir>/checkpoints/last.pt`).

**Run order:** object → event → attribute → multitask. Each is independent.

In [ ]:
import os, subprocess, time
env = os.environ.copy()
env['HF_HOME'] = HF_CACHE
env['PYTHONUNBUFFERED'] = '1'  # stream subprocess stdout to the cell live

RUNS = ['object', 'event', 'attribute', 'multitask']
RESULTS_BASE = f'{DRIVE_OUT}/results/track1_v2'   # v2 outputs separate from v1
for fam in RUNS:
    print(f'\n=== Training v2: {fam} ===', flush=True)
    t0 = time.time()
    out_dir = f'{RESULTS_BASE}/{fam}'
    cmd = [
        'python', '-u', 'train.py',
        '--config', f'configs/track1_{fam}.yaml',
        '--output-dir', out_dir,
        '--dataset-root', DATASET_ROOT,
        '--json-path', JSON_PATH,
        '--num-workers', '2',
    ]
    last_pt = f'{out_dir}/checkpoints/last.pt'
    if os.path.isfile(last_pt):
        print(f'   (resuming from {last_pt})', flush=True)
        cmd += ['--resume', last_pt]
    subprocess.run(cmd, cwd=REPO_DIR, env=env, check=True)
    print(f'=== {fam} done in {(time.time() - t0)/60:.1f} min ===', flush=True)

print('\n✅ All four v2 trainings complete.')

## 7. Evaluation on the test split + qualitative figures

Writes per-class CSVs, summary JSON, full prediction matrix, and ≥20 qualitative example PNGs + a combined PDF per run.

In [ ]:
import os, subprocess
env = os.environ.copy()
env['HF_HOME'] = HF_CACHE
env['PYTHONUNBUFFERED'] = '1'

RESULTS_BASE = f'{DRIVE_OUT}/results/track1_v2'

def run(cmd, label):
    print(f'  → {label}', flush=True)
    subprocess.run(cmd, cwd=REPO_DIR, env=env, check=True)

for fam in ['object', 'event', 'attribute', 'multitask']:
    out_dir = f'{RESULTS_BASE}/{fam}'
    ckpt = f'{out_dir}/checkpoints/best.pt'
    if not os.path.isfile(ckpt):
        print(f'(skip {fam}: no best.pt)', flush=True)
        continue
    print(f'\n=== Phase 2 eval pipeline: {fam} ===', flush=True)

    common = ['--config', f'configs/track1_{fam}.yaml',
              '--ckpt', ckpt,
              '--output-dir', out_dir,
              '--dataset-root', DATASET_ROOT,
              '--json-path', JSON_PATH]

    # Step 1: eval on val → dumps predictions_val.pt + eval_val.json + per_class_*_val.csv
    run(['python', '-u', 'eval.py'] + common + ['--split', 'val'],
        '1/6 eval val (flat 0.5)')

    # Step 2: tune per-class thresholds on val
    pred_val = f'{out_dir}/metrics/predictions_val.pt'
    thr_json = f'{out_dir}/metrics/thresholds_val.json'
    run(['python', '-u', '-m', 'src.scripts.tune_thresholds',
         '--predictions', pred_val, '--output', thr_json],
        '2/6 per-class threshold sweep on val')

    # Step 3: eval on test, flat 0.5 (baseline numbers; for direct v1↔v2 comparison)
    run(['python', '-u', 'eval.py'] + common + ['--split', 'test'],
        '3/6 eval test (flat 0.5)')

    # Step 4: eval on test with val-tuned per-class thresholds (final reportable numbers)
    run(['python', '-u', 'eval.py'] + common + ['--split', 'test', '--thresholds', thr_json],
        '4/6 eval test (val-tuned thresholds)')

    # Step 5: qualitative figures at flat 0.5 (matches v1 figures for direct comparison)
    run(['python', '-u', 'visualize.py'] + common + ['--split', 'test', '--num-samples', '20'],
        '5/6 visualize 20 samples (flat 0.5 — qualitative/)')

    # Step 6: qualitative figures using val-tuned thresholds (matches tuned metrics)
    run(['python', '-u', 'visualize.py'] + common + ['--split', 'test', '--num-samples', '20',
                                                       '--thresholds', thr_json],
        '6/6 visualize 20 samples (val-tuned — qualitative_tuned/)')

print('\n✅ Phase 2 eval + visualize complete.')
print(f'\nResults under {RESULTS_BASE}/<fam>/')
print('  metrics/eval_val.json         — val flat baseline')
print('  metrics/eval_test.json        — test flat baseline (compare to v1)')
print('  metrics/eval_test_tuned.json  — test with val-tuned per-class thresholds (final)')
print('  metrics/thresholds_val.json   — per-class thresholds picked on val')
print('  metrics/per_class_<fam>_{val,test,test_tuned}.csv')
print('  qualitative/                  — figures at flat 0.5')
print('  qualitative_tuned/            — figures using val-tuned thresholds')

## 8. Pin requirements for submission

Captures the actual library versions used in this Colab session into `requirements.txt`. This is what the course rubric requires.

Copy the output into the local repo's `requirements.txt` and commit before zipping the submission.

In [ ]:
!pip freeze | grep -iE '^(torch|torchvision|timm|scikit-learn|PyYAML|tqdm|numpy|matplotlib|Pillow|requests)=='

# Track 2 — v3 (A3 Hybrid): Cleanup + LDAM-DRW + cRT + LA + TTA

This section runs ONLY for the v3 Track-2 attempt. Skip if you're only doing Track 1.

Pipeline (per `docs/track2.md` § Alternative 3):
1. Predict v1 multi-task model on TRAIN split → `predictions_train.pt`.
2. Cleanlab-style noise ranking → `noisy_candidates.json`.
3. Gemini-relabel top-500 train candidates + entire val (1190 samples).
4. Conservative AND-rule reconciliation → `dataset_v3_clean.json`.
5. Retrain multitask on cleaned data with **LDAM-DRW + cRT**.
6. Eval with **logit adjustment + 4-way TTA + per-class threshold tuning**.

Total compute: ~10 min CPU steps + ~3 h A100 training + ~10 min eval.
API cost: ~$3 (Gemini 2.5 Flash, 1500 calls at ~1500 tokens each).

## v3 step 1 — Gemini API key

Stash your API key once in Colab's Secrets manager (left sidebar 🔑 icon, key name `GEMINI_API_KEY`). The next cell will pull it into the environment for the scripts.

In [ ]:
# Pull GEMINI_API_KEY from Colab Secrets into the env (or fall back to .env in /content).
import os
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    print('GEMINI_API_KEY loaded from Colab Secrets.')
except Exception as e:
    # Fall back: read from /content/deep-learning-project/.env if present.
    env_path = '/content/deep-learning-project/.env'
    if os.path.isfile(env_path):
        for line in open(env_path):
            if line.startswith('GEMINI_API_KEY='):
                os.environ['GEMINI_API_KEY'] = line.split('=', 1)[1].strip()
                print(f'GEMINI_API_KEY loaded from {env_path}.')
                break
    else:
        raise RuntimeError(f'No GEMINI_API_KEY found. Set it in Colab Secrets or {env_path}. ({e})')

## v3 step 2 — Install the cleanup-pipeline extras

In [ ]:
!pip install -q 'cleanlab>=2.6' 'google-genai>=1.0' 'python-dotenv>=1.0' 'pydantic>=2.0'

## v3 step 3 — Predict v1 multi-task on TRAIN split

We re-use the existing `results/track1_v1/multitask/checkpoints/best.pt` to dump logits + targets on the training set.

In [ ]:
import os, subprocess
env = os.environ.copy()
env['HF_HOME'] = HF_CACHE
env['PYTHONUNBUFFERED'] = '1'

V3_BASE = f'{DRIVE_OUT}/results/track1_v3'
CLEANUP_DIR = f'{V3_BASE}/cleanup'
V1_CKPT = f'{DRIVE_OUT}/results/track1_v1/multitask/checkpoints/best.pt'
os.makedirs(CLEANUP_DIR, exist_ok=True)

subprocess.run([
    'python', '-u', '-m', 'src.scripts.predict_for_cleanlab',
    '--config', 'configs/track1_multitask.yaml',
    '--ckpt', V1_CKPT,
    '--output-dir', CLEANUP_DIR,
    '--dataset-root', DATASET_ROOT,
    '--json-path', JSON_PATH,
    '--batch-size', '128',
    '--num-workers', '2',
], cwd=REPO_DIR, env=env, check=True)
print(f'predictions_train.pt → {CLEANUP_DIR}/')

## v3 step 4 — Rank training samples by likely label noise

In [ ]:
subprocess.run([
    'python', '-u', '-m', 'src.scripts.find_noisy_labels',
    '--predictions', f'{CLEANUP_DIR}/predictions_train.pt',
    '--output', f'{CLEANUP_DIR}/noisy_candidates.json',
    '--top-k', '500',
], cwd=REPO_DIR, env=env, check=True)

## v3 step 5 — Gemini relabel: top-500 train candidates + entire val

Two calls. Each writes a JSONL (append-only) so an API interruption can resume with `--resume`. Expected cost ~$3 total, ~30 min wall clock.

In [ ]:
# Train candidates (500 samples)
subprocess.run([
    'python', '-u', '-m', 'src.scripts.gemini_relabel',
    '--dataset-json', JSON_PATH,
    '--dataset-root', DATASET_ROOT,
    '--candidates', f'{CLEANUP_DIR}/noisy_candidates.json',
    '--output', f'{CLEANUP_DIR}/gemini_relabels_train.jsonl',
    '--max-samples', '500',
    '--resume',
], cwd=REPO_DIR, env=env, check=True)

In [ ]:
# Entire validation split (1190 samples)
subprocess.run([
    'python', '-u', '-m', 'src.scripts.gemini_relabel',
    '--dataset-json', JSON_PATH,
    '--dataset-root', DATASET_ROOT,
    '--split', 'val',
    '--output', f'{CLEANUP_DIR}/gemini_relabels_val.jsonl',
    '--resume',
], cwd=REPO_DIR, env=env, check=True)

## v3 step 6 — Reconcile cleaned dataset (conservative AND-rule)

In [ ]:
subprocess.run([
    'python', '-u', '-m', 'src.scripts.reconcile_labels',
    '--dataset-json', JSON_PATH,
    '--train-relabels', f'{CLEANUP_DIR}/gemini_relabels_train.jsonl',
    '--val-relabels',   f'{CLEANUP_DIR}/gemini_relabels_val.jsonl',
    '--output',         f'{CLEANUP_DIR}/dataset_v3_clean.json',
    '--summary',        f'{CLEANUP_DIR}/reconcile_summary.json',
], cwd=REPO_DIR, env=env, check=True)
import json
print(json.dumps(json.load(open(f'{CLEANUP_DIR}/reconcile_summary.json')), indent=2))

## v3 step 7 — Train multitask v3 on the cleaned dataset

LDAM-DRW + cRT + bf16 autocast. ~3 hours A100.

In [ ]:
V3_OUT = f'{V3_BASE}/multitask'
os.makedirs(V3_OUT, exist_ok=True)
cmd = [
    'python', '-u', 'train.py',
    '--config', 'configs/track2_v3_multitask.yaml',
    '--output-dir', V3_OUT,
    '--dataset-root', DATASET_ROOT,
    '--json-path', f'{CLEANUP_DIR}/dataset_v3_clean.json',
    '--num-workers', '2',
]
last_pt = f'{V3_OUT}/checkpoints/last.pt'
if os.path.isfile(last_pt):
    print(f'(resuming from {last_pt})')
    cmd += ['--resume', last_pt]
subprocess.run(cmd, cwd=REPO_DIR, env=env, check=True)
print('\n✅ v3 training complete (Phase A + cRT).')

## v3 step 8 — Eval v3 (val → tune thresholds → test with LA+TTA+thresholds)

We evaluate THREE flavours of v3 on test for ablation:
- flat 0.5 (baseline)
- + logit adjustment (LA)
- + LA + TTA + val-tuned per-class thresholds (final reportable)

In [ ]:
# Pick the better of best.pt (Phase A best) or crt_best.pt (cRT best).
import os
crt_best = f'{V3_OUT}/checkpoints/crt_best.pt'
ckpt = crt_best if os.path.isfile(crt_best) else f'{V3_OUT}/checkpoints/best.pt'
print(f'Using checkpoint: {ckpt}')

common = ['--config', 'configs/track2_v3_multitask.yaml',
          '--ckpt', ckpt,
          '--output-dir', V3_OUT,
          '--dataset-root', DATASET_ROOT,
          '--json-path', f'{CLEANUP_DIR}/dataset_v3_clean.json']
log_priors = f'{V3_OUT}/metrics/log_priors.json'

# Eval on cleaned val (flat) — needed for threshold tuning
subprocess.run(['python', '-u', 'eval.py'] + common + ['--split', 'val'],
              cwd=REPO_DIR, env=env, check=True)

# Tune per-class thresholds on cleaned val
subprocess.run(['python', '-u', '-m', 'src.scripts.tune_thresholds',
                '--predictions', f'{V3_OUT}/metrics/predictions_val.pt',
                '--output',      f'{V3_OUT}/metrics/thresholds_val.json'],
              cwd=REPO_DIR, env=env, check=True)
thresholds = f'{V3_OUT}/metrics/thresholds_val.json'

# Test (flat 0.5, no LA, no TTA) — baseline number
subprocess.run(['python', '-u', 'eval.py'] + common + ['--split', 'test'],
              cwd=REPO_DIR, env=env, check=True)

# Test + Logit Adjustment only
subprocess.run(['python', '-u', 'eval.py'] + common + ['--split', 'test',
                '--logit-adjust', log_priors, '--logit-adjust-tau', '1.0'],
              cwd=REPO_DIR, env=env, check=True)
os.rename(f'{V3_OUT}/metrics/eval_test_tuned.json',
          f'{V3_OUT}/metrics/eval_test_LA.json')

# Test + LA + TTA + val-tuned thresholds (final reportable)
subprocess.run(['python', '-u', 'eval.py'] + common + ['--split', 'test',
                '--logit-adjust', log_priors, '--logit-adjust-tau', '1.0',
                '--tta', '4',
                '--thresholds', thresholds],
              cwd=REPO_DIR, env=env, check=True)
os.rename(f'{V3_OUT}/metrics/eval_test_tuned.json',
          f'{V3_OUT}/metrics/eval_test_FINAL.json')

# Visualize 20 qualitative samples with tuned thresholds
subprocess.run(['python', '-u', 'visualize.py'] + common + ['--split', 'test',
                '--num-samples', '20', '--thresholds', thresholds],
              cwd=REPO_DIR, env=env, check=True)

print('\n✅ v3 evaluation complete.')
print(f'  eval_test.json        — flat 0.5 baseline')
print(f'  eval_test_LA.json     — + logit adjustment only')
print(f'  eval_test_FINAL.json  — + LA + TTA + val-tuned thresholds (report this)')